## Data Sources

| Source | Usage |
|---|---|
| World Bank Indicators API | GDP, GDP growth, population, internet users, services/industry value added |
| Google Trends CSV export | Search interest for project keywords |
| User/project data | traction, competition level, sector, country |

World Bank API documentation: https://datahelpdesk.worldbank.org/knowledgebase/articles/889392-about-the-indicators-api-documentation

Google Trends CSV export documentation: https://support.google.com/trends/answer/4365538

In [ ]:
from pathlib import Path
import json

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from market_data_collector import MarketDataCollector
from market_analysis_score_engine import MarketAnalysisScoreEngine

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 100)

## 1. Load Project Inputs

This file is not the market dataset. It is only a list of example project inputs used to call the external data sources.

In production, these inputs will come from MongoDB / Spring Boot when the entrepreneur submits a project.

In [ ]:
projects_path = Path("data/market_projects_sample.csv")
projects_df = pd.read_csv(projects_path)
projects_df

## 2. Collect Real Market Signals

The collector calls the World Bank API for each country and builds a dataset of market indicators.

If the notebook is executed without internet access, some values can be missing. That is normal. In that case, rerun this cell when connected, or keep the generated CSV cache from a previous execution.

In [ ]:
collector = MarketDataCollector(timeout_seconds=30)

market_df = collector.collect_dataset(
    projects_df.to_dict(orient="records"),
    output_path="data/market_signals_dataset.csv",
    use_world_bank=True,
)

market_df

## 3. Inspect Collected Features

The generated dataset contains both scoring features and external indicators.

Main scoring features:

- `market_size_billion`
- `market_growth_rate_percent`
- `competition_level`
- `product_traction_users`
- `search_trend_score`

External context features:

- `gdp_current_usd`
- `gdp_growth_percent`
- `population_total`
- `internet_users_percent`
- `services_value_added_percent_gdp`
- `industry_value_added_percent_gdp`
- `urban_population_percent`

In [ ]:
market_df.info()
market_df.describe(include="all")

## 4. Verify Data Sources

Each row keeps a JSON field named `data_sources`. This is important for reporting and for MLOps traceability.

In [ ]:
sample_sources = json.loads(market_df.loc[0, "data_sources"])
sample_sources.keys(), sample_sources["world_bank"][:2]

## 5. Optional: Add Google Trends CSV

Google Trends can be exported manually as CSV. Save files in `data/google_trends/`.

Example path: `data/google_trends/immobilier_morocco.csv`

The collector converts the exported interest-over-time values into `search_trend_score`.

In [ ]:
trend_csv = Path("data/google_trends/immobilier_morocco.csv")

if trend_csv.exists():
    trend_score = collector.read_google_trends_score(trend_csv)
    print(f"Google Trends score: {trend_score:.2f}")
else:
    print("No Google Trends CSV found yet. This is optional for now.")

## 6. Score Market Attractiveness

Now we use the collected dataset as input for the scoring engine.

The score is explainable and weighted:

| Feature | Weight |
|---|---:|
| Market size | 25% |
| Market growth | 25% |
| Competition | 20% |
| Traction | 15% |
| Search trend | 10% |
| Geographic fit | 5% |

In [ ]:
engine = MarketAnalysisScoreEngine()

results = []
for row in market_df.to_dict(orient="records"):
    result = engine.analyze(row)
    results.append({
        "project_name": row["project_name"],
        "sector": row["sector"],
        "country": row["country"],
        "market_score": result.market_score,
        "market_label": result.market_label,
        "confidence_score": result.confidence_score,
        **result.sub_scores,
    })

scores_df = pd.DataFrame(results)
scores_df

## 7. Visualizations

In [ ]:
plt.figure(figsize=(10, 5))
sns.barplot(data=scores_df, x="market_score", y="project_name", hue="sector", dodge=False)
plt.title("Market attractiveness score by project")
plt.xlabel("Market score")
plt.ylabel("Project")
plt.xlim(0, 100)
plt.legend(loc="lower right")
plt.show()

In [ ]:
sub_score_cols = ["market_size", "growth_rate", "competition", "traction", "trend", "geographic_fit"]
scores_df.set_index("project_name")[sub_score_cols].plot(kind="bar", figsize=(12, 5))
plt.title("Market sub-scores")
plt.ylabel("Score")
plt.ylim(0, 100)
plt.xticks(rotation=30, ha="right")
plt.show()

## 8. Save Final Market Dataset

This CSV can be used by the Streamlit app, FastAPI, and MLOps tracking.

In [ ]:
final_df = market_df.merge(
    scores_df[["project_name", "market_score", "market_label", "confidence_score"]],
    on="project_name",
    how="left",
)

final_path = Path("data/market_signals_scored_dataset.csv")
final_df.to_csv(final_path, index=False)
final_path, final_df.head()

## 9. What To Say In The Project Defense

Correct explanation:

> The Market Analysis module is not a supervised ML classifier because there is no universal labelled dataset for the success of every market and every business idea. Instead, we collect real market indicators from public data sources, enrich them with project-specific inputs, and compute an explainable market attractiveness score.

This is not fake ML. It is a data-driven scoring system. It becomes stronger when connected to more APIs or commercial datasets.